In [52]:
import pandas as pd

df = pd.read_csv('./openings.csv')


In [53]:
import pandas as pd

def transform_chess_dataset(df, target_openings, n_continuations=3):
    """
    Transforms the dataset into a vertex and edge dataframe.
    Handles 'skipping' moves and ensures black-move starts use <num>... notation.
    Assumes notation format like '1.e4' (no space after dot).
    """
    
    def get_parent_move_string(move_str):
        if not move_str or move_str == "Start":
            return None
        parts = move_str.strip().split(' ')
        parts.pop()
        # Handle cases where the last element was a move number/dot (e.g., "2.")
        if parts and parts[-1].endswith('.'):
            parts.pop()
        return " ".join(parts) if parts else "Start"

    # --- 1 & 3. Identify Nodes ---
    nodes_to_keep = set()
    for move_str in target_openings:
        parts = move_str.split(' ')
        for i in range(1, len(parts) + 1):
            prefix = " ".join(parts[:i])
            if not prefix.endswith('.'):
                nodes_to_keep.add(prefix)
    
    # --- 4. Identify Top N Continuations ---
    continuations = set()
    for target in target_openings:
        mask = df['Moves'].str.startswith(target + ' ')
        potential = df[mask].copy()
        potential['parent_check'] = potential['Moves'].apply(get_parent_move_string)
        top_n = (potential[potential['parent_check'] == target]
                 .sort_values('Num Games', ascending=False)
                 .head(n_continuations))
        continuations.update(top_n['Moves'].tolist())

    nodes_to_keep.update(continuations)

    # --- 1. Create Vertex Dataframe ---
    v_df = df[df['Moves'].isin(nodes_to_keep)].copy()
    start_row = pd.DataFrame({
        'Opening': ['Starting Position'],
        'Moves': ['Start'],
        'Num Games': [df['Num Games'].sum()]
    })
    v_df = pd.concat([start_row, v_df], ignore_index=True)

    for col in ['white_ideas', 'black_ideas', 'white_risks', 'black_risks', 'to_study']:
        v_df[col] = ""

    # --- 2 & 7. Create Edge Dataframe ---
    move_to_name = dict(zip(v_df['Moves'], v_df['Opening']))
    edges_data = []
    
    for _, row in v_df.iterrows():
        dst_move = row['Moves']
        dst_opening = row['Opening']
        
        if dst_move == "Start":
            continue
            
        # Recursive search for the nearest ancestor
        curr_parent = get_parent_move_string(dst_move)
        while curr_parent is not None and curr_parent not in move_to_name:
            curr_parent = get_parent_move_string(curr_parent)
        
        if curr_parent in move_to_name:
            src_opening = move_to_name[curr_parent]
            
            if curr_parent == "Start":
                move_played = dst_move
            else:
                # Get the string that represents the moves skipped/played
                move_played = dst_move[len(curr_parent):].strip()
                
                # Check if the sequence starts with a Black move.
                # Since notation is '1.e4', a Black move will NOT start with a digit.
                if move_played and not move_played[0].isdigit():
                    # Find the last move number in the parent string. 
                    # If parent is '1.e4', we find '1'.
                    parent_parts = curr_parent.split(' ')
                    last_full_move = [p for p in parent_parts if '.' in p][-1]
                    move_num = last_full_move.split('.')[0]
                    
                    move_played = f"{move_num}...{move_played}"

            edges_data.append({
                'src': src_opening,
                'dst': dst_opening,
                'move': move_played
            })

    e_df = pd.DataFrame(edges_data)
    return v_df, e_df

In [57]:
targets = [
    "1.e4 e5 2.Nf3 Nc6 3.Bc4", # Italian
    "1.e4 e5 2.Nf3 Nc6 3.Bb5", # Spanish
    "1.e4 e5 2.Nf3 Nc6 3.Nc3 Nf6", # Four knights
    "1.e4 e5 2.Nf3 Nf6", # Russian
    "1.d4 d5 2.Nf3 Nf6", # Queen's pawn symmetrical
    "1.e4 d5 2.exd5", # Scandi
    "1.e4 e6", # French
    "1.e4 c6 2.d4 d5", # Caro
    "1.e4 c5", # Sicilian
    "1.e4 e5 2.Nf3 d6 3.d4" # Philidor
]
vertices, edges = transform_chess_dataset(df, targets, n_continuations=5)

In [61]:
import os
v_cols = ['name', 'white_ideas', 'black_ideas', 'white_risks', 'black_risks', 'to_study', 'moves']

vertices = vertices.rename(columns={"Opening": "name", "Moves": "moves"})
edges = edges.rename(columns={"move": "added_moves"})

if os.path.exists("./vertices.csv"):
    old_vertices = pd.read_csv("./vertices.csv").rename(columns={"Opening": "name", "Moves": "moves"})
    old_edges = pd.read_csv("./edges.csv").rename(columns={"move": "added_moves"})
    
    vertices = pd.concat([old_vertices, vertices]).drop_duplicates("name").sort_values('name')
    edges = pd.concat([old_edges, edges]).drop_duplicates(["src", "dst"]).sort_values(['src', 'dst'])
    
vertices = vertices[v_cols]
    
# Check if there are existing vertices and edges. If so, just add the new rows
# I.e. drop duplicates on name and src


In [64]:
vertices.to_csv('vertices.csv', index=False)
edges.to_csv('edges.csv', index=False)